# Workflow 06

## Configuration

In [2]:
chemfilesdir = "chemfiles/"
resultsdir = "results/"
docked = resultsdir + "docked.sdf"
log = resultsdir + "gninalog.txt"
rmsdlog = resultsdir + "rmsd.txt"
masterlog = resultsdir + "masterlog.txt"

In [3]:
experimentdescription = "Place experiment description here."

In [4]:
proteindict = {}
liganddict = {}
gninaoptdict = {}
prepdict = {}

## Select protein and it's complexed ligand

In [5]:
# Protein Name Source Database PDB ID Domain CNN Score
# Acetylcholinesterase PDB-REDO 6O4W B 0.92
# https://pdb-redo.eu/db/6o4w
# Binary complex of native hAChE with Donepezil
#cif = "workflow03/6o4w_final.cif"
#!wget -nv -O "workflow03/6o4w_final.cif" "https://pdb-redo.eu/db/6o4w/6o4w_final.cif"
#Some of the packages don't work well with the newer .cif file format.'
# TEAM - do we want to update our workflow to utilize the new mmCIF (.cif) file format?
redopdb = chemfilesdir + "6o4w_final_redo.pdb"
!wget -nv -O "{redopdb}" "https://pdb-redo.eu/db/6o4w/6o4w_final.pdb"

2026-01-13 11:19:59 URL:https://pdb-redo.eu/db/6o4w/6o4w_final.pdb [733809] -> "chemfiles/6o4w_final_redo.pdb" [1]


In [6]:
# Get same protein from RCBS database: https://www.rcsb.org/structure/6O4W
rcbspdb = chemfilesdir + "6O4w_rcbs.pdb"
#!wget -nv -O "workflow03/6O4w_rcbs.pdb" "https://files.rcsb.org/download/6O4W.pdb"
!wget -nv -O "{rcbspdb}" "https://files.rcsb.org/download/6O4W.pdb"

2026-01-13 11:20:27 URL:https://files.rcsb.org/download/6O4W.pdb [775980] -> "chemfiles/6O4w_rcbs.pdb" [1]


## Identify bound ligand(s)

In [ ]:
# What ligands are in our .cif file? Many. These two blocks identify the 'code' and chain.
# code is E20, chains are A and B
# Confirmation of code and chains below:
# https://proteopedia.org/wiki/index.php/6o4w confirms ligand codes.
# https://www.ebi.ac.uk/pdbe/entry/pdb/6o4w?activeTab=ligands&id=E20
# https://www.rcsb.org/ligand-validation/6O4W/E20
#
# We are assuming that the '604' value below is the sequence id which is likely specific to this .cif.
# Here are the final 6 columns header information and one line from a HETATM row:
#_atom_site.auth_seq_id 
#_atom_site.auth_comp_id 
#_atom_site.auth_asym_id 
#_atom_site.auth_atom_id 
#_atom_site.pdbx_PDB_model_num 
#_atom_site.pdbx_tls_group_id 
#604 E20 A C28 1 ?

#TEAM - what is the best way to identify the bound ligand? Code or online database? Database method might not scale.

In [7]:
import gemmi

structure = gemmi.read_structure(chemfilesdir + "6o4w_final_redo.pdb")
#structure = gemmi.read_structure(chemfilesdir + "6O4w_rcbs.pdb")
ligands = []

for model in structure:
    for chain in model:
        for res in chain:
            if res.het_flag != ' ':  # hetero-residue
                if res.name not in ("HOH", "WAT", "H2O"):
                    if res.seqid.num == 604:
                        ligands.append((res.name, chain.name, res.seqid.num))

sorted(set(ligands))

[('E20', 'A', 604), ('E20', 'B', 604)]

## Remove Chain 'A'

In [8]:
from Bio.PDB import PDBParser, PDBIO, Select

class KeepNonAChains(Select):
    def accept_chain(self, chain):
        # Return False for chain A, True for all others
        return chain.id != "A"

in_pdb = chemfilesdir + "6o4w_final_redo.pdb"
out_pdb = chemfilesdir + "6o4w_final_redo_chainB.pdb"

parser = PDBParser(QUIET=True)
structure = parser.get_structure("struct", in_pdb)

io = PDBIO()
io.set_structure(structure)
io.save(out_pdb, KeepNonAChains())


In [9]:
in_pdb = chemfilesdir + "6O4w_rcbs.pdb"
out_pdb = chemfilesdir + "6O4w_rcbs_chainB.pdb"

parser = PDBParser(QUIET=True)
structure = parser.get_structure("struct", in_pdb)

io = PDBIO()
io.set_structure(structure)
io.save(out_pdb, KeepNonAChains())


## Extract the bound ligand

In [ ]:
# This block creates two identical files.
# the -L flag does not seem to care what 'label' it is passed.
# I beleive the .cif has at least three ligands so I am not confident this code creates a ligand file with only E20.
# It appears the '-l' flag is wholly ignored. The output file type seems to instruct obabel how it should modify the molecule.

#!obabel "workflow03/6o4w_final.cif" -l "LIG" -O "workflow03/6o4w_final_ligand.sdf"
#!obabel "workflow03/6o4w_final.cif" -l "E20" -osdf -O "workflow03/6o4w_final_ligand_E20.sdf"
#!obabel "workflow03/6o4w_final_chainB.pdb" -l "E20" -osdf -O "workflow03/6o4w_final_chainB_ligand_E20.sdf"

In [10]:
# Extract ligand based on ligand code which we know is E20 from literature AND from code in the 'Identify Ligand'
# block earlier in this file.

from Bio.PDB import PDBParser, PDBIO, Select

class LigandSelect(Select):
    def __init__(self, ligand_name, waters=None):
        self.ligand_name = ligand_name.upper()
        self.waters = waters or {"HOH", "WAT", "H2O"}

    def accept_residue(self, residue):
        hetflag, resseq, icode = residue.id
        resname = residue.get_resname().strip().upper()

        # Hetero residue with the desired name, not water
        if hetflag != " " and resname == self.ligand_name and resname not in self.waters:
            return True
        return False

    def accept_atom(self, atom):
        # Keep all atoms of accepted residues
        return True

def extract_ligand_by_resname(in_pdb, out_pdb, ligand_code):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("struct", in_pdb)

    io = PDBIO()
    io.set_structure(structure)
    io.save(out_pdb, LigandSelect(ligand_code))


# Call previous functions. 
# Notice that output file is still in .pdb format. Not ideal.
if __name__ == "__main__":
    extract_ligand_by_resname(
        in_pdb = chemfilesdir + "6o4w_final_redo_chainB.pdb",
        out_pdb = chemfilesdir + "6o4w_final_redo_chainB_E20.pdb",
        ligand_code = "E20",   # residue/ligand code
    )
if __name__ == "__main__":
    extract_ligand_by_resname(
        in_pdb= chemfilesdir + "6O4w_rcbs_chainB.pdb",
        out_pdb= chemfilesdir + "6O4w_rcbs_chainB_E20.pdb",
        ligand_code = "E20",   # residue/ligand code
    )

In [11]:
# Convert ligands files from .pdb to .sdf
# This preparation block is labeled 'rdkit save H'
from rdkit import Chem

mol = Chem.MolFromPDBFile(chemfilesdir + "6o4w_final_redo_chainB_E20.pdb", removeHs=False) #removeHS=False preserves all hydrogens
w = Chem.SDWriter(chemfilesdir + "6o4w_final_redo_chainB_E20.sdf")
w.write(mol)
w.close()

mol = Chem.MolFromPDBFile(chemfilesdir + "6O4w_rcbs_chainB_E20.pdb", removeHs=False) #removeHS=False preserves all hydrogens
w = Chem.SDWriter(chemfilesdir + "6O4w_rcbs_chainB_E20.sdf")
w.write(mol)
w.close()

prep = {}
prep["name"] = "rdkit_save_H"
prep["description"] = "Python 'Bio.PDB' package used to remove the 'A' chain from the protein file."
prep["description"] += " Python 'Bio.PDB' package used to extract the ligand via code name."
prep["description"] += " Python 'rdkit' package used to convert the ligand file from PDB file to SDF file format."
prep["functionname"] = "TBD"
prepdict["rdkit_save_H"] = prep

In [13]:
# Define ligands and add to liganddict

ligand = {}
ligand["code"] = "E20"
ligand["name"] = "E20 extracted from protein 6o4w_final_redo_chainB.pdb"
ligand["filename"] = "6o4w_final_redo_chainB_E20.sdf"
ligand["source"] = "Extracted from protein file 6o4w_final_redo_chainB.pdb"
ligand["prep"] = "rdkit_save_H"
liganddict["REDOcomplexedE20"] = ligand

ligand = {}
ligand["code"] = "E20"
ligand["name"] = "E20 extracted from protein 6O4w_rcbs_chainB_E20.pdb"
ligand["filename"] = "6O4w_rcbs_chainB_E20.sdf"
ligand["prep"] = "rdkit_save_H"
liganddict["RCBScomplexedE20"] = ligand

In [ ]:
# Extract ligand via a method supplied by Maya
# This seems to create a ligand that returns the same results as the ligand created above.
#!awk '$1=="HETATM" && $4=="E20" && $5=="B"' workflow03/ligand_E20.pdb > workflow03/ligand_E20_maya.pdb

## Create test ligand from SMILES

In [14]:
# Prepare one or many ligands with Openbabel.
# Source: https://www.ebi.ac.uk/pdbe-srv/pdbechem/chemicalCompound/show/E20#structures-section
smilessdf = chemfilesdir + "E20.sdf"
!obabel -:"COc1cc2c(cc1OC)C(=O)[C@@H](C2)CC3CCN(CC3)Cc4ccccc4" --gen3d -osdf -O"{smilessdf}"

1 molecule converted


In [15]:
ligand = {}
ligand["code"] = "E20"
ligand["name"] = "E20 created from SMILES"
ligand["filename"] = "E20.sdf"
ligand["prep"] = "obabel --gen3d"
liganddict["smilesE20"] = ligand

In [16]:
prep = {}
prep["name"] = "obabel --gen3d"
prep["description"] = "OpenBabel executable used to convert SMILES chain into 3D .sdf file via '--gen3d' flag."
prep["functionname"] = "TBD"
prepdict["obabel --gen3d"] = prep

## Fix the protein

In [17]:
#PDBFixer python package is required by the following block. 
#This code expects that PDBFixer package is installed in the Conda environment that Jupyter is running within.
from pdbfixer import PDBFixer
from openmm.app import PDBFile

In [18]:
# define the PDBFixer function
# Options must be called in this specific order. Some options rely on earlier options.
# Andy provided PDBFixer suggestions in 1/17/2025 email.
# Remove chains – no. As far as I can tell there is not a standard set of chains that ought to be removed.
# Identify missing residues – yes
# replace non-standard residues – yes
# remove heterogens – yes. we want to remove the water molecules for GNINA and I think the command will remove all, including water
# add missing heavy atoms – yes
# add missing hydrogens – yes. use the default pH of 7. It’d probably be more accurate to use slightly higher, though I would suspect that Kim just uses the default.
# Add water – no. I am curious to test this function because the reaction would take place in a solvent but not yet sure that GNINA accepts this.
# Add membrane – no. Similar comment as above.
# https://htmlpreview.github.io/?https://github.com/openmm/pdbfixer/blob/master/Manual.html
# mode: 0 = rewrite no tasks, 1 = removeHeterogens only, 2 = Also hydrogens, 3 = full set of Andy tasks, 4 full set minus Hydrogen

def PDBfixfunc(localprotein, savelocation, mode = 0):

    fixer = PDBFixer(filename = localprotein)

    if (mode >= 2 ):
      #fixer.removeChains(indices) - never
      fixer.findMissingResidues()

    if (mode >= 3 ):
      fixer.findNonstandardResidues()
      fixer.replaceNonstandardResidues()

    if (mode == 1 or mode >= 3 ):
      fixer.removeHeterogens(False) #removes the complexed ligand
      # The argument specifies whether to keep water molecules. 
      # False removes all heterogens including water. True keeps water molecules while removing all other heterogens. 

    if (mode >= 2 and mode <= 3 ):
      fixer.findMissingAtoms() # required by addMissingAtoms
      fixer.addMissingAtoms() # required by addMissingHydrogens
      fixer.addMissingHydrogens(7.0)
      #fixer.addSolvent() - never
      #fixer.addMembrane('POPE') - never

    #PDBFile.writeFile(fixer.topology, fixer.positions, open(savelocation, 'w'))
    PDBFile.writeFile(fixer.topology, fixer.positions, open(savelocation, 'w'), keepIds=True)

    del(fixer)


In [19]:
prep = {}
prep["name"] = "PDBfixfunc"
prep["description"] = "Python 'Bio.PDB' package used to remove the 'A' chain from the protein file."
prep["description"] += " PDBFixer package used to fix the protein with following options:"
prep["description"] += " findMissingResidues(), findNonstandardResidues(), replaceNonstandardResidues(), fixer.removeHeterogens(False),"
prep["description"] += " findMissingAtoms(), addMissingAtoms(), addMissingHydrogens(7.0)"
prep["functionname"] = "TBD"
prepdict["PDBfixfunc"] = prep

In [21]:
#PDBFix with one of four modes
PDBfixfunc(chemfilesdir + "6o4w_final_redo_chainB.pdb",chemfilesdir + "6o4w_final_redo_chainB_fixed.pdb",3)

protein = {}
protein["code"] = "6o4w"
protein["name"] = "6o4w from RDBSREDO"
protein["filename"] = "6o4w_final_redo_chainB_fixed.pdb"
protein["prep"] = "PDBfixfunc"
proteindict ["PDB-REDO6o4w"] = protein

In [22]:
PDBfixfunc(chemfilesdir + "6O4w_rcbs_chainB.pdb",chemfilesdir + "6O4w_rcbs_chainB_fixed.pdb",3) 

protein = {}
protein["code"] = "6o4w"
protein["name"] = "6o4w from RCBS"
protein["filename"] = "6O4w_rcbs_chainB_fixed.pdb"
protein["prep"] = "PDBfixfunc"
proteindict ["RCBS6o4w"] = protein

## Setup scoring function and results structure

In [23]:
# Get values from line 1 of results section from GNINA results file.
def get_mode1_values(path):
    mode_section = False
    with open(path) as f:
        mode_section = False
        for line in f:
            line = line.strip()
            if line.startswith("mode"):
                mode_section = True
            if not line or line.startswith("mode") or line.startswith("-") or line.startswith("|") or mode_section is False:
                continue
            parts = line.split()
            mode = int(parts[0])
            if mode == 1:
                affinity = float(parts[1])
                intramol = float(parts[2])
                cnn_pose = float(parts[3])
                cnn_aff = float(parts[4])
                return mode, affinity, intramol, cnn_pose, cnn_aff
# Example usage:   mode, affinity, intramol, cnn_pose, cnn_aff = get_mode1_values("workflow03/log.txt")

In [24]:
opt = "--exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu" 

gninaopt = {}
gninaopt["opt"] = opt
gninaoptdict["ex16nm8seed0CNN--no_gpu"] = gninaopt

## GNINA and reporting functions

In [ ]:
#Examples
#output = !~/gnina-binary/gnina -r proteins_pdbfixtest/2ama_fixer_andy_minusH.pdb -l proteins_pdbfixtest/Trenbolone.sdf --autobox_ligand proteins_pdbfixtest/2ama-ligand.pdb -o docked.sdf.gz  --exhaustiveness=64 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --log proteins_pdbfixtest/log.txt
# Predict against complexed ligand.
#!~/octoberproject/gnina -r 8GUTciffixed4.pdb -l 8GUT-ligand.sdf --autobox_ligand 8GUT-ligand.sdf -o docked.sdf.gz  --exhaustiveness=64 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --log log.txt --no_gpu  

In [26]:
# Call Gnina, parse results from RMSD calculation, call function that parse results from both log files and appends to results variable

#pro = subdir + "6O4w_rcbs_chainB_fixed.pdb"
#lig = subdir + "6O4w_rcbs_chainB_E20.sdf"
#box = subdir + "6O4w_rcbs_chainB_E20.sdf"
#docked = subdir + "docked.sdf"
#log = subdir + "gninalog.txt"
#!~/octoberproject/gnina -r "{pro}" -l "{lig}" --autobox_ligand "{box}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu  

def rungnina(proteinid,ligandid,boxid):
    protein = proteindict[proteinid]
    ligand = liganddict[ligandid]
    box = liganddict[boxid]
    p = chemfilesdir + protein["filename"]
    l = chemfilesdir + ligand["filename"]
    b = chemfilesdir + box["filename"]
    !~/octoberproject/gnina -r "{p}" -l "{l}" --autobox_ligand "{b}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu  

    #Compute RMSD of two docked ligand files and save results to rmsdlog file.
    !obrms -f "{b}" "{docked}" | tee "{rmsdlog}"

    appendresults(proteinid,ligandid,boxid)

In [27]:
# Get values from text files, append to results data, print to screen.
# Note: The CNN_VS score is also report in the docked.sdf file.

def appendresults(pro, lig, box):

    mode, affinity, intramol, cnnpose, cnnaffinity = get_mode1_values(log)
    cnnvs = cnnpose * cnnaffinity
    
    with open(rmsdlog) as f:
        first_line = f.readline()
        line = first_line.strip()
        parts = line.split()
        rmsd = float(parts[2])
    new_row = (pro + " + " + lig + " @ " + box, cnnpose, cnnvs, rmsd)
    data.append(new_row)

In [28]:
# Report results as array of lines
import copy

def reportdict(rows, columns):

    lines = []

    if len(rows) == 0:
        return (lines) 
    
    # 1. Build ordered list of columns: first column is the outer key (call it "id"), remaining columns are all inner keys discovered from the data.
    inner_keys = set()
    for r in rows.values():
        inner_keys.update(r.keys())
    #columns = ["id"] + sorted(inner_keys)
    #OR specify explicetly
    #columns = ["id", "name", "prep"]
    # 2. Compute column widths using both headers and data
    col_widths = {}
    # width for "id" column
    col_widths["id"] = max(len("id"), max(len(k) for k in rows))
    # widths for inner columns
    for col in inner_keys:
        header_len = len(col)
        data_len = max(len(str(r.get(col, ""))) for r in rows.values())
        col_widths[col] = max(header_len, data_len)
    # 3. Print header
    header = "  ".join(f"{col:<{col_widths[col]}}" for col in columns)
    lines.append(header)
    #print(header)
    #print("-" * len(header))
    # 4. Print rows
    for outer_key, inner in rows.items():
        cells = [f"{outer_key:<{col_widths['id']}}"]
        for col in columns[1:]:
            cells.append(f"{str(inner.get(col, '')):<{col_widths[col]}}")
        #print("  ".join(cells))
        lines.append("  ".join(cells))

    return (lines)

In [29]:
def reporttable(rows):
    lines = []
    # rows: list of tuples, all same length
    if not rows:
        return
    # Number of columns
    n_cols = len(rows[0])
    # Max width per column (over all rows)
    widths = [
        max(len(str(row[col])) for row in rows)
        for col in range(n_cols)
    ]
    # Print each row with aligned columns
    for row in rows:
        line = "  ".join(
            f"{str(value):<{widths[i]}}"
            for i, value in enumerate(row)
        )
        #print(line)
        lines.append(line)

    return (lines)

In [ ]:
reporttable(data)

In [30]:
#Report out results with details of Ligands and Proteins used
from datetime import datetime

now = datetime.now()
stamp = now.strftime("%Y-%m-%d %H:%M:%S")
unixtime = int(datetime.now().timestamp())

def reportall():
    lines = []

    lines.append("Report produced " + stamp + " experiment id: " + str(unixtime))

    lines.append("")
    lines.append(experimentdescription)

    lines.append("")
    cols = ["name","functionname","description"]
    lines.append("File preparation descriptions:")
    lines = lines + reportdict(prepdict,cols)
                
    lines.append("")
    cols = ["id", "name", "prep"]
    lines.append("Proteins:")
    lines = lines + reportdict(proteindict, cols)

    lines.append("")
    lines.append("Ligands:")
    lines = lines + reportdict(liganddict, cols)

    lines.append("")
    cols = ["opt"]
    lines.append("Gnina options:")
    lines = lines + reportdict(gninaoptdict,cols)

    lines.append("")
    lines.append("Results:")
    newrow = ("proteinid + ligandid @ ligandid_searchspace", "CNN Score", "CNN_VS", "RMSD") 
    outputlist = list(data)
    outputlist.insert(0, newrow)
    lines = lines + reporttable(outputlist)

    return lines;

In [ ]:
out = reportall()
print (out)


## Call Gnina

In [31]:
#populate first row with comparison data from Bucchari paper.
data = [
    ("Bucchari (PDB-REDO)", "0.92", "7.29", "1.71"),
]

In [32]:
#1
#rungnina("protein_id", "ligand_id", "ligand_id_for_searchspace")
rungnina("PDB-REDO6o4w", "REDOcomplexedE20", "REDOcomplexedE20")

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r chemfiles/6o4w_final_redo_chainB_fixed.pdb -l chemfiles/6o4w_final_redo_chainB_E20.sdf --autobox_ligand chemfiles/6o4w_final_redo_chainB_E20.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol) | pose score | affinity
-----+------------+------------+

In [33]:
#2
#rungnina("protein_id", "ligand_id", "ligand_id_for_searchspace")
rungnina("PDB-REDO6o4w", "smilesE20", "REDOcomplexedE20")

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r chemfiles/6o4w_final_redo_chainB_fixed.pdb -l chemfiles/E20.sdf --autobox_ligand chemfiles/6o4w_final_redo_chainB_E20.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol) | pose score | affinity
-----+---------

In [34]:
#3
#rungnina("protein_id", "ligand_id", "ligand_id_for_searchspace")
rungnina("RCBS6o4w", "RCBScomplexedE20", "RCBScomplexedE20")

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r chemfiles/6O4w_rcbs_chainB_fixed.pdb -l chemfiles/6O4w_rcbs_chainB_E20.sdf --autobox_ligand chemfiles/6O4w_rcbs_chainB_E20.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol) | pose score | affinity
-----+------------+------------+------------+-----

## Display all results

In [35]:
#This block presents the results to stdout.
lines = reportall()
text = "\n".join(lines) + "\n"
print(text)

Report produced 2026-01-13 11:38:46 experiment id: 1768333126

Place experiment description here.

File preparation descriptions:
name            functionname  description                                                                                                                                                                                                                                                                                                                 
rdkit_save_H    TBD           Python 'Bio.PDB' package used to remove the 'A' chain from the protein file. Python 'Bio.PDB' package used to extract the ligand via code name. Python 'rdkit' package used to convert the ligand file from PDB file to SDF file format.                                                                                    
obabel --gen3d  TBD           OpenBabel executable used to convert SMILES chain into 3D .sdf file via '--gen3d' flag.                                                           

In [36]:
#This block will append results listed above to master results file.

with open(masterlog, "a") as f:
    f.write(text)

## Maya Results from 12/15/2025

In [ ]:
https://colab.research.google.com/drive/14LTM5RHdjNY8h49LJ7C5K5_l5B_IqyTa?usp=drive_link#scrollTo=8mTtZuPU5E8Z

Results:

Method	                                CNN ScoreCNN_VS	RMSD
Bucchari (PDB-REDO)	                    0.92	7.29	1.71
PDB-REDO file with re-docking Donepezil	0.96	7.14	0.39
PDB-REDO file with generated Donepezil	0.94	6.85	1.07
RCSB PDB file with re-docking Donepezil	0.93	6.85	0.58


## Notes

In [ ]:
#!jupyter nbconvert --to html --stdout workflow02.ipynb
!jupyter nbconvert --to html workflow02.ipynb